# MISTIC kernel comparison vs Extra Trees playground

Use this notebook to compare RBF MISTIC, mixed linear–RBF MISTIC with shared features, mixed linear–RBF MISTIC with kernel-specific selection units, and Extra Trees on paired blind folds across synthetic classification scenarios. All MISTIC variants use the unified `greedy_forward_selection` API and `combined_rank`; the separate mixed case selects `(kernel_id, perturbation_set)` units independently. Edit the configuration cell, inspect the resolved experiment, and then enable `RUN_EXPERIMENT`. Results are checkpointed under a hash of the complete configuration, so changing a setting cannot silently reuse incompatible fits.

The known signal mask is used only after prediction to evaluate feature recovery; it is never supplied to either model.

In [1]:
from dataclasses import asdict, replace
from hashlib import sha256
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

from mistic.examples.classification_benchmark_suite import (
    BenchmarkConfig, DatasetSpec, IndependentMixedMisticClassifier, MisticClassifier,
    MixedMisticClassifier, _safe_split_count,
    dataset_specs, evaluate_method, feature_stability, make_dataset,
    sklearn_methods,
)

sns.set_theme(style='whitegrid', context='notebook')
EXAMPLE_DIR = Path.cwd()
if EXAMPLE_DIR.name != 'examples':
    EXAMPLE_DIR = Path('mistic/examples')
OUTPUT_ROOT = EXAMPLE_DIR / 'mistic_extra_trees_playground_results'

Matplotlib is building the font cache; this may take a moment.


## Experiment configuration

Available scenarios are `clean_linear`, `correlated`, `nonlinear`, and `noisy_imbalanced`. Set medoids or maximum features to `None` to use `mistic_feature_fraction` of dimensionality. `MISTIC_MAX_SELECTION_FEATURES` counts kernel-feature assignments in the separate mixed case, whereas reported `selected_feature_count` is the number of unique original columns. For a linear kernel, gamma settings are ignored.

In [2]:
# Dataset design
SCENARIOS = ('correlated', 'nonlinear', 'noisy_imbalanced')
SAMPLE_SIZES = (50,)
FEATURE_COUNTS = (100,)
DATA_SEEDS = (2026, 2027, 2028)
OUTER_FOLDS = 3
INNER_FOLDS = 3

# Current preferred MISTIC configuration
MISTIC_MEMBERS = 5
MISTIC_RANK_WEIGHT = 0.90
MISTIC_FEATURE_FRACTION = 0.20
MISTIC_NUM_FEATURE_MEDOIDS = 50  # e.g. 20, or None for fraction scaling
MISTIC_MAX_FEATURES = None          # e.g. 20, or None for fraction scaling
MISTIC_CLASS_WEIGHT = True
MISTIC_TUNE_EACH_STEP = True
MISTIC_ADDITION_FACTOR = 0.20
MISTIC_SCORE_WEIGHT = 0.50          # AUC weight; F1 weight is 1 - this
MISTIC_KERNEL = 'rbf'               # 'rbf' or 'linear'

# Hyperparameter candidates
MISTIC_PARAMETER_STRATEGY = 'latin_hypercube'  # or 'grid'
MISTIC_LHS_TRIALS = 10
MISTIC_C_BOUNDS = (0.1, 8.0)
MISTIC_GAMMA_BOUNDS = (1e-10, 4)
MISTIC_C_VALUES = (0.25, 1.0, 4.0)
MISTIC_GAMMA_VALUES = (2**-9, 2**-7, 2**-5)
MISTIC_MIXED_LINEAR_WEIGHTS = (0.5,)  # (0.25, 0.5, 0.75)
MISTIC_MAX_CANDIDATE_FEATURES = 40
MISTIC_MAX_SELECTION_FEATURES = 40  # kernel-feature assignments for the separate mixed case
MISTIC_NUM_INITIAL_SELECTION_UNITS = 1

RANDOM_SEED = 42
RUN_EXPERIMENT = False  # Change to True when ready
FORCE_RERUN = False     # True overwrites checkpoints for this exact config

In [3]:
config = BenchmarkConfig(
    sample_sizes=SAMPLE_SIZES, feature_counts=FEATURE_COUNTS,
    scenarios=SCENARIOS, data_seeds=DATA_SEEDS,
    outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS,
    mistic_members=MISTIC_MEMBERS,
    mistic_rank_weight=MISTIC_RANK_WEIGHT,
    mistic_feature_fraction=MISTIC_FEATURE_FRACTION,
    mistic_num_feature_medoids=MISTIC_NUM_FEATURE_MEDOIDS,
    mistic_max_features=MISTIC_MAX_FEATURES,
    mistic_class_weight=MISTIC_CLASS_WEIGHT,
    mistic_tune_models_each_step=MISTIC_TUNE_EACH_STEP,
    mistic_addition_factor=MISTIC_ADDITION_FACTOR,
    mistic_score_weight=MISTIC_SCORE_WEIGHT,
    mistic_parameter_strategy=MISTIC_PARAMETER_STRATEGY,
    mistic_kernel=MISTIC_KERNEL,
    mistic_lhs_trials=MISTIC_LHS_TRIALS,
    mistic_c_bounds=MISTIC_C_BOUNDS,
    mistic_gamma_bounds=MISTIC_GAMMA_BOUNDS,
    mistic_c_values=MISTIC_C_VALUES,
    mistic_gamma_values=MISTIC_GAMMA_VALUES,
    mistic_mixed_linear_weights=MISTIC_MIXED_LINEAR_WEIGHTS,
    mistic_max_candidate_features=MISTIC_MAX_CANDIDATE_FEATURES,
    mistic_max_selection_features=MISTIC_MAX_SELECTION_FEATURES,
    mistic_num_initial_selection_units=MISTIC_NUM_INITIAL_SELECTION_UNITS,
    random_seed=RANDOM_SEED,
)

payload = json.dumps(asdict(config), sort_keys=True, default=list)
CONFIG_ID = sha256(payload.encode()).hexdigest()[:12]
RUN_DIR = OUTPUT_ROOT / CONFIG_ID
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(asdict(config), indent=2))

resolved = pd.DataFrame([
    {
        'n_features': p,
        'medoids': MISTIC_NUM_FEATURE_MEDOIDS or max(1, round(MISTIC_FEATURE_FRACTION*p)),
        'max_unique_features': MISTIC_MAX_FEATURES or max(1, round(MISTIC_FEATURE_FRACTION*p)),
        'max_candidate_features': min(p, MISTIC_MAX_CANDIDATE_FEATURES),
        'max_selection_features': min(2 * p, MISTIC_MAX_SELECTION_FEATURES),
    } for p in FEATURE_COUNTS
])
print(f'Configuration: {CONFIG_ID}')
print(f'Datasets: {len(SCENARIOS)*len(SAMPLE_SIZES)*len(FEATURE_COUNTS)*len(DATA_SEEDS)}')
print(f'Output: {RUN_DIR.resolve()}')
display(resolved)

Configuration: 0f025d571e8b
Datasets: 9
Output: /Users/chriskieslich/Library/CloudStorage/Dropbox-GaTech/Christopher Kieslich/006_Code/interpretable-explainable-svms/mistic/examples/mistic_extra_trees_playground_results/0f025d571e8b


,n_features,medoids,max_unique_features,max_candidate_features,max_selection_features
0,100,50,20,40,40


## Paired benchmark runner

All four methods use the same stratified outer folds. Every completed dataset is written immediately. The MISTIC wrappers share the unified forward-selection entry point; only their kernel expression and shared-versus-independent unit routing differ. Extra Trees is configured identically to the main benchmark.

In [4]:
def benchmark_pair(spec, config):
    X, y, signal_mask = make_dataset(spec)
    n_outer = _safe_split_count(y, config.outer_folds)
    outer = StratifiedKFold(
        n_outer, shuffle=True,
        random_state=config.random_seed + spec.data_seed,
    )
    rows = []
    for fold, (train, test) in enumerate(outer.split(X, y)):
        seed = config.random_seed + 1000*spec.data_seed + fold
        inner = _safe_split_count(y.iloc[train], config.inner_folds)
        methods = {
            'mistic_unified': MisticClassifier(config, seed),
            'mistic_mixed': MixedMisticClassifier(config, seed),
            'mistic_mixed_independent': IndependentMixedMisticClassifier(config, seed),
            'extra_trees': sklearn_methods(seed, inner)['extra_trees'],
        }
        for name, estimator in methods.items():
            try:
                result = {
                    'status': 'ok', 'error_type': None, 'error_message': None,
                    **evaluate_method(
                        name, estimator, X.iloc[train], y.iloc[train],
                        X.iloc[test], y.iloc[test], signal_mask, seed,
                    ),
                }
            except Exception as error:
                result = {
                    'method': name, 'status': 'failed',
                    'error_type': type(error).__name__,
                    'error_message': str(error)[:1000],
                }
            rows.append({**asdict(spec), 'outer_fold': fold, **result})
    return pd.DataFrame(rows)


def run_paired_benchmark(config, force=False):
    completed = []
    specs = dataset_specs(config)
    for index, spec in enumerate(specs, start=1):
        path = CHECKPOINT_DIR / f'{spec.name}.csv'
        if path.exists() and not force:
            frame = pd.read_csv(path)
            state = 'cached'
        else:
            frame = benchmark_pair(spec, config)
            frame.to_csv(path, index=False)
            state = 'completed'
        completed.append(frame)
        print(f'[{index:>3}/{len(specs)}] {state}: {spec.name}', flush=True)
    results = pd.concat(completed, ignore_index=True)
    results.to_csv(RUN_DIR / 'all_results.csv', index=False)
    return results

In [5]:
if RUN_EXPERIMENT:
    results = run_paired_benchmark(config, force=FORCE_RERUN)
elif (RUN_DIR / 'all_results.csv').exists():
    results = pd.read_csv(RUN_DIR / 'all_results.csv')
    print('Loaded cached results.')
else:
    results = pd.DataFrame()
    print('Ready. Set RUN_EXPERIMENT=True and rerun from the configuration cell.')

Ready. Set RUN_EXPERIMENT=True and rerun from the configuration cell.


## Results and diagnostics

In [6]:
METRICS = [
    'roc_auc', 'pr_auc', 'f1', 'balanced_accuracy', 'mcc',
    'signal_recall_at_k', 'feature_average_precision',
    'selected_feature_count', 'elapsed_seconds',
]

if not results.empty:
    failures = results[results.status != 'ok'][
        ['name', 'outer_fold', 'method', 'error_type', 'error_message']
    ]
    if len(failures):
        display(failures)
    successful = results[results.status == 'ok'].copy()
    overall = successful.groupby('method')[METRICS].agg(['mean', 'std', 'count'])
    scenario_summary = successful.groupby(['scenario', 'method'])[METRICS].agg(['mean', 'std', 'count'])
    overall.to_csv(RUN_DIR / 'overall_summary.csv')
    scenario_summary.to_csv(RUN_DIR / 'scenario_summary.csv')
    display(overall)
    display(scenario_summary)
    

In [7]:
if not results.empty:
    labels = {
        'mistic_unified': 'MISTIC (RBF)',
        'mistic_mixed': 'MISTIC (linear + RBF)',
        'mistic_mixed_independent': 'MISTIC (linear + RBF, separate)',
        'extra_trees': 'Extra Trees',
    }
    successful['Model'] = successful.method.map(labels)
    plot_metrics = {
        'roc_auc': 'ROC AUC',
        'pr_auc': 'PR AUC',
        'balanced_accuracy': 'Balanced accuracy',
        'signal_recall_at_k': 'Signal recall@20',
        'feature_average_precision': 'Feature average precision',
    }
    long = successful.melt(
        id_vars=['scenario', 'n_samples', 'n_features', 'Model'],
        value_vars=list(plot_metrics), var_name='metric', value_name='value',
    )
    long['metric'] = long.metric.map(plot_metrics)
    grid = sns.catplot(
        data=long, x='Model', y='value', hue='Model',
        row='scenario', col='metric', kind='bar', errorbar=('ci', 95),
        sharey=False, height=2.8, aspect=1.6, legend=False,
    )
    grid.set_axis_labels('', 'Blind-fold score')
    grid.set_titles(row_template='{row_name}', col_template='{col_name}')
    grid.figure.suptitle('MISTIC kernel comparison vs Extra Trees by scenario', y=1.01)
    grid.figure.savefig(RUN_DIR / 'scenario_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()

In [8]:
if not results.empty:
    scaling = successful.groupby(
        ['scenario', 'n_samples', 'n_features', 'Model'], as_index=False
    )[['roc_auc', 'signal_recall_at_k', 'feature_average_precision']].mean()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for axis, metric, title in zip(
        axes,
        ['roc_auc', 'signal_recall_at_k', 'feature_average_precision'],
        ['ROC AUC', 'Signal recall@20', 'Feature average precision'],
    ):
        sns.lineplot(
            data=scaling, x='n_samples', y=metric, hue='Model',
            style='n_features', markers=True, dashes=False, ax=axis,
        )
        axis.set_xscale('log')
        axis.set_title(title)
        axis.set_xlabel('Samples (log scale)')
    fig.suptitle('Scaling across selected scenarios and dimensions', y=1.02)
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'scaling_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()

    stability = feature_stability(successful)
    stability.to_csv(RUN_DIR / 'feature_stability.csv', index=False)
    display(stability)

## Comparing experiments

Each configuration receives its own directory. Use the cell below after running multiple configurations to compare their saved summaries without mixing checkpoints.

In [9]:
experiment_rows = []
for result_path in sorted(OUTPUT_ROOT.glob('*/all_results.csv')):
    frame = pd.read_csv(result_path)
    frame = frame[frame.status == 'ok'].copy()
    frame['configuration'] = result_path.parent.name
    experiment_rows.append(frame)

if experiment_rows:
    experiments = pd.concat(experiment_rows, ignore_index=True)
    comparison = experiments.groupby(['configuration', 'method'])[METRICS].mean()
    display(comparison)
else:
    print('No completed playground experiments yet.')

roc_auc    pr_auc        f1  \
configuration method                                                   
0bf3db90fc4d  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.491117  0.475745  0.408398   
108d26e64ce7  extra_trees               0.576719  0.566260  0.329465   
              mistic_unified            0.616735  0.563653  0.448790   
1110e23a5bba  extra_trees               0.570186  0.539891  0.374234   
              mistic_mixed              0.533904  0.530001  0.419023   
              mistic_mixed_independent  0.532655  0.493664  0.390716   
              mistic_unified            0.557536  0.522733  0.424228   
47cb4f5b155e  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.493128  0.459629  0.345563   
6a87fbe47665  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.535334  0.480258  0.400142   
816cd97557f8  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.508375  0.501754  0.428028   
8d022ceee560  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.481704  0.459514  0.393504   
a6d26784b795  extra_trees               0.570186  0.539891  0.374234   
              mistic_mixed              0.535104  0.481684  0.400768   
              mistic_mixed_independent  0.499601  0.467127  0.379532   
              mistic_unified            0.535334  0.480258  0.400142   
ac8901a39b36  extra_trees               0.570186  0.539891  0.374234   
              mistic_mixed              0.547762  0.529974  0.429316   
              mistic_mixed_independent  0.555586  0.493982  0.393739   
              mistic_unified            0.527333  0.484465  0.408067   
b97a86f5c419  extra_trees               0.570186  0.539891  0.374234   
              mistic_mixed              0.535104  0.481684  0.400768   
              mistic_mixed_independent  0.539841  0.510970  0.439214   
              mistic_unified            0.535334  0.480258  0.400142   
c8cb5dde67ac  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.541441  0.508512  0.443624   
cd0eea41bdd2  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.533955  0.516812  0.454225   
d084fb043bc2  extra_trees               0.570186  0.539891  0.374234   
              mistic_mixed              0.535104  0.481684  0.400768   
              mistic_unified            0.535334  0.480258  0.400142   
d18b4f065a18  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.540160  0.503240  0.408407   
e1c0d9325a20  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.538669  0.503979  0.397648   
ea372fc94ecf  extra_trees               0.570186  0.539891  0.374234   
              mistic_unified            0.539693  0.518811  0.468286   

                                        balanced_accuracy       mcc  \
configuration method                                                  
0bf3db90fc4d  extra_trees                        0.567836  0.141062   
              mistic_unified                     0.528549  0.057348   
108d26e64ce7  extra_trees                        0.548611  0.104183   
              mistic_unified                     0.557685  0.113231   
1110e23a5bba  extra_trees                        0.567836  0.141062   
              mistic_mixed                       0.524799  0.065550   
              mistic_mixed_independent           0.520797  0.047419   
              mistic_unified                     0.514002  0.031738   
47cb4f5b155e  extra_trees                        0.567836  0.141062   
              mistic_unified                     0.495845 -0.002889   
6a87fbe47665  extra_trees                        0.567836  0.141062   
              mistic_